In [1]:
import numpy as np
import torch
import torch.nn as nn
from deep_hedging_env import HedgingEnv
from logit_normal import LogitNormal
from reward_utils import compute_discounted_cumsum_rewards
from plot_utils import plot_portfolio_vs_option_price

In [2]:
S0 = np.array([50.0, 100.0, 200.0])
K = np.array([[45.0, 55.0], [90.0, 110.0], [180.0, 220.0]])
sigma = np.array([0.15, 0.2, 0.25])
r = 0.05
num_simulation = 5
num_step = 250

# Perfect Delta Hedging

In [3]:
env = HedgingEnv(S0, K, sigma, r, num_simulation=num_simulation, num_step=num_step)
(
    num_simulation,
    num_asset,
    num_strike,
) = (
    env.num_simulation,
    env.num_asset,
    env.num_strike,
)
state, _ = env.reset(seed=0)
idx = 0
while True:
    action = env.ground_truth_deltas[..., idx].reshape(-1)
    _, _, done, _, _ = env.step(action.astype(np.float32))
    if all(done):
        break
    else:
        idx += 1

In [4]:
np.abs(env.portfolio_value[...,1:]-env.option_prices[...,1:]).mean()

np.float64(0.25130805741869644)

# Random Policy

In [5]:
env = HedgingEnv(S0, K, sigma, r, num_simulation=num_simulation, num_step=num_step)
(
    num_simulation,
    num_asset,
    num_strike,
) = (
    env.num_simulation,
    env.num_asset,
    env.num_strike,
)
state, _ = env.reset(seed=0)
while True:
    action = env.np_random.uniform(low=0.0, high=1.0, size=num_simulation * num_asset * num_strike)
    _, _, done, _, _ = env.step(action.astype(np.float32))
    if all(done):
        break

In [6]:
np.abs(env.portfolio_value[...,1:]-env.option_prices[...,1:]).mean()

np.float64(5.629992543156087)

# Rule-based Policy

In [7]:
class PolicyNetwork(nn.Module):
    def __init__(
        self,
    ):
        super(PolicyNetwork, self).__init__()

    def forward(self, history_features):
        action = history_features[:, -1, 6] # action is the estimated delta

        return action, None

In [8]:
# --- Env. Parameters ---
S0 = np.array([50.0, 100.0, 200.0])
K = np.array([[45.0, 55.0], [90.0, 110.0], [180.0, 220.0]])
sigma = np.array([0.15, 0.2, 0.25])
r = 0.05
num_simulation = 5
num_step = 250

env = HedgingEnv(
    S0, K, sigma, r, num_simulation=num_simulation, num_step=num_step
)

# --- Policy Network Parameters ---
history_len = 15

policy_net = PolicyNetwork()

In [9]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
policy_net.to(device)

for episode in range(10):
    log_prob_history = []
    reward_history = []
    state_history = []

    state, _ = env.reset(seed=1)
    state = state[:, None, :]
    state_history.append(state)

    while True:
        policy_net_input = np.concatenate(state_history[-history_len:], axis=1)
        policy_net_input = torch.tensor(policy_net_input, dtype=torch.float32).to(
            device
        )
        action, log_prob = policy_net(policy_net_input)
        action_np = action.detach().cpu().numpy()
        state, reward, done, _, _ = env.step(action_np)
        reward_history.append(reward)
        state = state[:, None, :]
        state_history.append(state)

        if all(done):
            break

    print(f"Episode {episode + 1}, Avg. Reward: {np.array(reward_history).mean()}")

Episode 1, Avg. Reward: -1.6022545334669067
Episode 2, Avg. Reward: -1.6022545334669067
Episode 3, Avg. Reward: -1.6022545334669067
Episode 4, Avg. Reward: -1.6022545334669067
Episode 5, Avg. Reward: -1.6022545334669067
Episode 6, Avg. Reward: -1.6022545334669067
Episode 7, Avg. Reward: -1.6022545334669067
Episode 8, Avg. Reward: -1.6022545334669067
Episode 9, Avg. Reward: -1.6022545334669067
Episode 10, Avg. Reward: -1.6022545334669067


In [10]:
np.abs(env.portfolio_value[...,1:]-env.option_prices[...,1:]).mean()

np.float64(1.6022545334669065)

In [11]:
plot_portfolio_vs_option_price(env)

# Policy Gradient (Simple MLP)

In [12]:
class PolicyNetwork(nn.Module):
    def __init__(
        self,
        input_dim,
        hidden_size,
        action_dim=1,
    ):
        super(PolicyNetwork, self).__init__()
        self.fc1 = nn.Linear(input_dim, hidden_size)
        self.fc_mu = nn.Linear(hidden_size, action_dim)
        self.fc_sigma = nn.Linear(hidden_size, action_dim)
        self.softplus = nn.Softplus()

    def forward(self, history_features):

        x = history_features[
            :, -1, :
        ]  # simple MLP just uses the latest state's feature
        x = torch.tanh(self.fc1(x))
        mu = self.fc_mu(x)
        sigma = self.softplus(self.fc_sigma(x))
        return mu, sigma

    def sample_action(self, mu, sigma, deterministic=False):
        logit_normal = LogitNormal(mu, sigma)

        if deterministic:
            action = torch.sigmoid(mu)
        else:
            action = logit_normal.sample()
        log_prob = logit_normal.log_prob(action)

        return action, log_prob

In [13]:
# --- Env. Parameters ---
S0 = np.array([50.0, 100.0, 200.0])
K = np.array([[45.0, 55.0], [90.0, 110.0], [180.0, 220.0]])
sigma = np.array([0.15, 0.2, 0.25])
r = 0.05
num_simulation = 100
num_step = 250

env = HedgingEnv(
    S0, K, sigma, r, num_simulation=num_simulation, num_step=num_step
)

# --- Policy Network Parameters ---
input_dim = 11
hidden_size = 64
history_len = 15

policy_net = PolicyNetwork(input_dim, hidden_size)

# --- Optimization Parameters ---
learning_rate = 1e-3

optimizer = torch.optim.Adam(policy_net.parameters(), lr=learning_rate)

# --- Other Parameters ---
num_episodes = 200
num_epochs = 10
discount_factor = 0.999

In [14]:
# Train

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
policy_net.to(device)

for epoch in range(num_epochs):
    for episode in range(num_episodes):
        log_prob_history = []
        reward_history = []
        state_history = []

        state, _ = env.reset(seed=epoch+1000) # avoid using seed=0 as it's for testing
        state = state[:, None, :]
        state_history.append(state)

        while True:
            policy_net_input = np.concatenate(state_history[-history_len:], axis=1)
            policy_net_input = torch.tensor(policy_net_input, dtype=torch.float32).to(
                device
            )
            action_mu, action_sigma = policy_net(policy_net_input)
            action, log_prob = policy_net.sample_action(action_mu, action_sigma)
            log_prob_history.append(log_prob)
            action_np = action.detach().cpu().numpy()
            state, reward, done, _, _ = env.step(action_np)
            reward_history.append(reward)
            state = state[:, None, :]
            state_history.append(state)

            if all(done):
                break

        R = compute_discounted_cumsum_rewards(np.array(reward_history), discount_factor)
        R = R - R.mean(axis=1, keepdims=True)
        R = R / (R.std(axis=1, keepdims=True) + np.finfo(R.dtype).eps)
        R = torch.tensor(R, dtype=torch.float32).to(device)
        optimizer.zero_grad()
        loss = (-R * torch.stack(log_prob_history)).mean()
        loss.backward()
        optimizer.step()

        if (episode + 1) % 10 == 0:
            print(
                f"Epoch {epoch+1}/{num_epochs}, Episode {episode + 1}/{num_episodes}, Loss: {loss.item()}, Avg. Reward: {np.array(reward_history).mean()}"
            )

Epoch 1/10, Episode 10/200, Loss: -0.0009212608565576375, Avg. Reward: -4.418090045540839
Epoch 1/10, Episode 20/200, Loss: -0.005893348250538111, Avg. Reward: -4.382055114142983
Epoch 1/10, Episode 30/200, Loss: -0.0063481200486421585, Avg. Reward: -4.163402241837833
Epoch 1/10, Episode 40/200, Loss: -0.007245361339300871, Avg. Reward: -4.079325060665173
Epoch 1/10, Episode 50/200, Loss: -0.0073019759729504585, Avg. Reward: -4.039068368258145
Epoch 1/10, Episode 60/200, Loss: -0.00995844230055809, Avg. Reward: -4.009046056954363
Epoch 1/10, Episode 70/200, Loss: -0.012703846208751202, Avg. Reward: -3.889283297313579
Epoch 1/10, Episode 80/200, Loss: -0.01371737476438284, Avg. Reward: -3.9055798874281775
Epoch 1/10, Episode 90/200, Loss: -0.01792007125914097, Avg. Reward: -3.6162868890542574
Epoch 1/10, Episode 100/200, Loss: -0.018831051886081696, Avg. Reward: -3.5991163351443047
Epoch 1/10, Episode 110/200, Loss: -0.02431350015103817, Avg. Reward: -3.4372529365203244
Epoch 1/10, Epis

In [15]:
action_sigma.min(), action_sigma.max(), action_sigma.mean() 

(tensor(0.0317, grad_fn=<MinBackward1>),
 tensor(1.0184, grad_fn=<MaxBackward1>),
 tensor(0.1838, grad_fn=<MeanBackward0>))

In [16]:
# Test

env = HedgingEnv(S0, K, sigma, r, num_simulation=5, num_step=num_step)

log_prob_history = []
reward_history = []
state_history = []

state, _ = env.reset(seed=0)
state = state[:, None, :]
state_history.append(state)

while True:
    policy_net_input = np.concatenate(state_history[-history_len:], axis=1)
    policy_net_input = torch.tensor(policy_net_input, dtype=torch.float32).to(device)
    action_mu, action_sigma = policy_net(policy_net_input)
    action, log_prob = policy_net.sample_action(action_mu, action_sigma, True)
    log_prob_history.append(log_prob)
    action_np = action.detach().cpu().numpy()
    state, reward, done, _, _ = env.step(action_np)
    reward_history.append(reward)
    state = state[:, None, :]
    state_history.append(state)

    if all(done):
        break

In [17]:
rewards = np.array(reward_history)
rewards.min(), rewards.max(), rewards.mean(), rewards.std()

(np.float64(-69.63491497504643),
 np.float64(-7.875589470884847e-05),
 np.float64(-6.133598410798962),
 np.float64(8.288506985071347))

In [18]:
plot_portfolio_vs_option_price(env)

# Policy Gradient (Recurrent)

In [19]:
class PolicyNetwork(nn.Module):
    def __init__(
        self,
        input_dim,
        hidden_size,
        num_layers,
        action_dim=1,
        history_len=5,
        dropout=0.0,
    ):
        super(PolicyNetwork, self).__init__()
        self.history_len = history_len
        self.rnn = nn.GRU(
            input_dim, hidden_size, num_layers, batch_first=True, dropout=dropout
        )
        self.fc_mu = nn.Linear(hidden_size, action_dim)
        self.fc_sigma = nn.Linear(hidden_size, action_dim)
        self.softplus = nn.Softplus()

    def forward(self, history_features, determistic=False):
        # history_features: (batch_size, history_len, feature_dim)

        batch_size = history_features.size(0)
        seq_len = history_features.size(1)

        # Pad history if shorter than history_len
        if seq_len < self.history_len:
            padding = torch.zeros(
                batch_size,
                self.history_len - seq_len,
                history_features.size(2),
                dtype=history_features.dtype,
                device=history_features.device,
            )
            history_features = torch.cat([padding, history_features], dim=1)

        output, _ = self.rnn(
            history_features
        )  # out: tensor of shape (batch_size, seq_length, hidden_size)
        output = output[:, -1, :]  # Take output from the last time step

        mu = self.fc_mu(output)
        sigma = self.softplus(self.fc_sigma(output))

        return mu, sigma

    def sample_action(self, mu, sigma, deterministic=False):
        logit_normal = LogitNormal(mu, sigma)

        if deterministic:
            action = torch.sigmoid(mu)
        else:
            action = logit_normal.sample()
        log_prob = logit_normal.log_prob(action)

        return action, log_prob

In [ ]:
# --- Env. Parameters ---
S0 = np.array([50.0, 100.0, 200.0])
K = np.array([[45.0, 55.0], [90.0, 110.0], [180.0, 220.0]])
sigma = np.array([0.15, 0.2, 0.25])
r = 0.05
num_simulation = 100
num_step = 250

env = HedgingEnv(
    S0, K, sigma, r, num_simulation=num_simulation, num_step=num_step
)

# --- Policy Network Parameters ---
input_dim = 11
hidden_size = 64
num_layers = 2
history_len = 15

policy_net = PolicyNetwork(input_dim, hidden_size, num_layers, history_len=history_len)

# --- Optimization Parameters ---
learning_rate = 1e-4

optimizer = torch.optim.Adam(policy_net.parameters(), lr=learning_rate)

# --- Other Parameters ---
num_episodes = 200
num_epochs = 10
discount_factor = 0.999

In [21]:
# Train

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
policy_net.to(device)

for epoch in range(num_epochs):
    for episode in range(num_episodes):
        log_prob_history = []
        reward_history = []
        state_history = []

        state, _ = env.reset(seed=epoch+1000) # avoid using seed=0 as it's for testing
        state = state[:, None, :]
        state_history.append(state)

        while True:
            policy_net_input = np.concatenate(state_history[-history_len:], axis=1)
            policy_net_input = torch.tensor(policy_net_input, dtype=torch.float32).to(
                device
            )
            action_mu, action_sigma = policy_net(policy_net_input)
            action, log_prob = policy_net.sample_action(action_mu, action_sigma)
            log_prob_history.append(log_prob)
            action_np = action.detach().cpu().numpy()
            state, reward, done, _, _ = env.step(action_np)
            reward_history.append(reward)
            state = state[:, None, :]
            state_history.append(state)

            if all(done):
                break

        R = compute_discounted_cumsum_rewards(np.array(reward_history), discount_factor)
        R = R - R.mean(axis=1, keepdims=True)
        R = R / (R.std(axis=1, keepdims=True) + np.finfo(R.dtype).eps)
        R = torch.tensor(R, dtype=torch.float32).to(device)
        optimizer.zero_grad()
        loss = (-R * torch.stack(log_prob_history)).mean()
        loss.backward()
        optimizer.step()

        if (episode + 1) % 10 == 0:
            print(
                f"Epoch {epoch+1}/{num_epochs}, Episode {episode + 1}/{num_episodes}, Loss: {loss.item()}, Avg. Reward: {np.array(reward_history).mean()}"
            )

Epoch 1/10, Episode 10/200, Loss: -0.009200439788401127, Avg. Reward: -5.017048272188787
Epoch 1/10, Episode 20/200, Loss: 0.0090932697057724, Avg. Reward: -4.469467408494522
Epoch 1/10, Episode 30/200, Loss: -0.0005553385126404464, Avg. Reward: -4.3093445282266485
Epoch 1/10, Episode 40/200, Loss: 0.0005932779749855399, Avg. Reward: -4.532445426520809
Epoch 1/10, Episode 50/200, Loss: 0.00242706504650414, Avg. Reward: -3.968246844076997
Epoch 1/10, Episode 60/200, Loss: 0.008108516223728657, Avg. Reward: -4.3983294275593146
Epoch 1/10, Episode 70/200, Loss: -0.006219108123332262, Avg. Reward: -3.9359249223302317
Epoch 1/10, Episode 80/200, Loss: -0.007496207486838102, Avg. Reward: -4.362046674003798
Epoch 1/10, Episode 90/200, Loss: -0.004152252338826656, Avg. Reward: -3.651143613100956
Epoch 1/10, Episode 100/200, Loss: -0.0027244058437645435, Avg. Reward: -4.345177528677597
Epoch 1/10, Episode 110/200, Loss: 2.229919482488185e-05, Avg. Reward: -4.6879251861632625
Epoch 1/10, Episode

In [22]:
action_sigma.min(), action_sigma.max(), action_sigma.mean() 

(tensor(0.5732, grad_fn=<MinBackward1>),
 tensor(0.6284, grad_fn=<MaxBackward1>),
 tensor(0.6043, grad_fn=<MeanBackward0>))

In [23]:
# Test

env = HedgingEnv(S0, K, sigma, r, num_simulation=5, num_step=num_step)

log_prob_history = []
reward_history = []
state_history = []

state, _ = env.reset(seed=0)
state = state[:, None, :]
state_history.append(state)

while True:
    policy_net_input = np.concatenate(state_history[-history_len:], axis=1)
    policy_net_input = torch.tensor(policy_net_input, dtype=torch.float32).to(device)
    action_mu, action_sigma = policy_net(policy_net_input)
    action, log_prob = policy_net.sample_action(action_mu, action_sigma, True)
    log_prob_history.append(log_prob)
    action_np = action.detach().cpu().numpy()
    state, reward, done, _, _ = env.step(action_np)
    reward_history.append(reward)
    state = state[:, None, :]
    state_history.append(state)

    if all(done):
        break

In [24]:
rewards = np.array(reward_history)
rewards.min(), rewards.max(), rewards.mean(), rewards.std()

(np.float64(-49.82883206762918),
 np.float64(-4.762600128316308e-05),
 np.float64(-3.7776630805340066),
 np.float64(5.7708594536604725))

In [25]:
plot_portfolio_vs_option_price(env)